# ADVC — Kaggle Notebook (Tiny-ImageNet, T4 GPU)

Runs top-to-bottom in **batch mode** (Save & Run All) so it keeps going with your PC off. Every phase is **resumable** — recorded CSV rows are skipped and trained levels reload via `--skip-training`.

### The one cell you edit each run: **Cell 5d — Batch Control Panel**
Save & Run All executes every cell, so you don't comment cells out. Instead Cell 5d has switches; each phase cell no-ops if not selected. Scope each batch to a chunk that fits ~12h.

### Before running — notebook Settings (right sidebar)
1. **Accelerator** → `GPU T4 x2` (never run on CPU)
2. **Internet** → `On`
3. **Add-ons → Secrets** → `HF_TOKEN`, and (for saving) `KAGGLE_USERNAME` + `KAGGLE_KEY`
4. **Add-ons → Datasets** → attach Tiny-ImageNet, AND (from batch 2 on) your `advc-results` dataset.

### Persistence
`/kaggle/working` is wiped between sessions. Cell 13 saves `results/` to the `advc-results` Kaggle dataset; attach it next batch and Cell 5b restores it.

In [ ]:
# Cell 1 — Verify GPU
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free, total = torch.cuda.mem_get_info(0)
    print('GPU name       :', name)
    print(f'Free VRAM      : {free/1e9:.1f} GB / {total/1e9:.1f} GB total')
    print('T4 confirmed.' if 'T4' in name else 'WARNING: expected Tesla T4')
else:
    print('WARNING: no GPU. Set Settings > Accelerator > GPU T4 x2, then restart.')

In [ ]:
# Cell 2 — HF token + dependencies
import os, subprocess, sys
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets.')
except Exception as e:
    print('WARNING: could not load HF_TOKEN secret:', e)

packages = ['timm', 'torchattacks', 'bitsandbytes', 'optimum', 'pyyaml',
            'tqdm', 'accelerate', 'huggingface_hub', 'transformers>=4.44.0,<5.0']
res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages,
                     capture_output=True, text=True)
print('pip error:\n' + res.stderr[-2000:] if res.returncode else 'Installed: ' + ', '.join(packages))

In [ ]:
# Cell 2b — Authenticate the kaggle CLI (needed for Cell 13 dataset save/create)
# One-time setup on Kaggle:
#   1. Profile -> Settings -> API -> 'Create New Token'  (downloads kaggle.json)
#   2. Notebook -> Add-ons -> Secrets -> add TWO secrets from that file:
#        KAGGLE_USERNAME = <username>   KAGGLE_KEY = <key>
# Without this, `kaggle datasets create/version` in Cell 13 fails with a 401.
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = _s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = _s.get_secret('KAGGLE_KEY')
    print('Kaggle CLI authenticated as', os.environ['KAGGLE_USERNAME'])
except Exception as e:
    print('WARNING: kaggle CLI not authenticated:', e)
    print('Add KAGGLE_USERNAME and KAGGLE_KEY in Add-ons > Secrets, '
          'or Cell 13 dataset save will fail (experiment still runs).')

In [ ]:
# Cell 3 — Clone / update repo, set cwd
import os, sys, subprocess
REPO_URL = 'https://github.com/Jmanav/ADVC.git'
REPO_DIR = '/kaggle/working/ADVC'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=False)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('cwd is now:', os.getcwd())
subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'])
assert os.path.exists('configs/base.yaml'), 'configs/base.yaml missing — clone failed?'

In [ ]:
# Cell 4 — Extract + prepare Tiny-ImageNet (read-only source, writable --out)
import os, glob, zipfile, subprocess, sys
SRC_ROOT = None
OUT_ROOT = '/kaggle/working/tiny_if'   # WRITABLE — matches configs/base.yaml
for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True):
    if os.path.isdir(p):
        SRC_ROOT = p
        break
if SRC_ROOT is None:
    zips = (glob.glob('/kaggle/input/**/tiny-imagenet-200.zip', recursive=True)
            or glob.glob('/kaggle/input/**/*tiny*imagenet*.zip', recursive=True))
    assert zips, 'No tiny-imagenet-200 folder or zip under /kaggle/input.'
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall('/kaggle/working/')
    SRC_ROOT = '/kaggle/working/tiny-imagenet-200'
print('Source:', SRC_ROOT, '\nOutput:', OUT_ROOT)
res = subprocess.run([sys.executable, 'scripts/prepare_tiny_imagenet.py',
                      '--root', SRC_ROOT, '--out', OUT_ROOT], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print('prepare failed:\n', res.stderr[-2000:])
else:
    for d in ['train_if', 'val_if']:
        path = os.path.join(OUT_ROOT, d)
        print(f'  {d}:', len(os.listdir(path)) if os.path.isdir(path) else 0, 'class folders')

In [ ]:
# Cell 5 — Output directories (dataset-scoped, matching the code paths)
import os
DS_NAME = 'tiny-imagenet'
for d in ['results', f'results/{DS_NAME}',
          f'results/checkpoints/{DS_NAME}/at', f'results/checkpoints/{DS_NAME}/atkd',
          f'results/{DS_NAME}/figures']:
    os.makedirs(d, exist_ok=True)
    print('Ready:', d)

In [ ]:
# Cell 5b — Restore prior results (CSVs + checkpoints) from the advc-results dataset
import os, glob, shutil
DS_NAME = 'tiny-imagenet'
os.makedirs(f'results/{DS_NAME}', exist_ok=True)
os.makedirs(f'results/checkpoints/{DS_NAME}/at', exist_ok=True)
os.makedirs(f'results/checkpoints/{DS_NAME}/atkd', exist_ok=True)

def _find(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None

restored_csv = 0
for csv_name in ['phase1_results.csv', 'phase2_at_results.csv',
                 'phase2_atkd_results.csv', 'phase3_results.csv']:
    src = _find(f'/kaggle/input/**/{DS_NAME}/{csv_name}')
    if src:
        shutil.copy(src, f'results/{DS_NAME}/{csv_name}')
        print('Restored CSV        ->', src)
        restored_csv += 1

restored_ck = 0
for sub in ['at', 'atkd']:
    for ckpt in glob.glob(f'/kaggle/input/**/checkpoints/{DS_NAME}/{sub}/*.pt', recursive=True):
        shutil.copy(ckpt, f'results/checkpoints/{DS_NAME}/{sub}/{os.path.basename(ckpt)}')
        restored_ck += 1
    n = len(os.listdir(f'results/checkpoints/{DS_NAME}/{sub}'))
    print(f'checkpoints/{DS_NAME}/{sub}: {n} file(s) present')

if restored_csv == 0 and restored_ck == 0:
    print('\nWARNING: nothing restored from /kaggle/input for', DS_NAME,
          '- if not your first batch, attach advc-results or everything recomputes.')

In [ ]:
# Cell 5c — Verify resume state (answers 'will it restart?' BEFORE spending GPU)
import os, glob, pandas as pd
DS_NAME = 'tiny-imagenet'
EPOCHS = 7  # --skip-training only reuses the FINAL epoch

print('=== CSV rows already recorded (these attack rows will be SKIPPED) ===')
for name in ['phase1_results.csv', 'phase2_at_results.csv',
             'phase2_atkd_results.csv', 'phase3_results.csv']:
    p = f'results/{DS_NAME}/{name}'
    if os.path.exists(p):
        df = pd.read_csv(p)
        combos = sorted(set(zip(df.get('compression', []), df.get('attack', []))))
        print(f'  {name}: {len(df)} rows -> {combos}')
    else:
        print(f'  {name}: MISSING (will compute fresh)')

print(f'\n=== Reusable checkpoints (epoch{EPOCHS:02d} = loadable via --skip-training) ===')
for sub in ['at', 'atkd']:
    for lvl in ['fp32', 'int8', 'int4']:
        hit = glob.glob(f'results/checkpoints/{DS_NAME}/{sub}/{sub}_{lvl}_epoch{EPOCHS:02d}*.pt')
        print(f'  {sub}/{lvl}:', ('REUSABLE ' + os.path.basename(hit[0])) if hit else 'none -> will TRAIN')

In [ ]:
# Cell 5d — BATCH CONTROL PANEL  (edit this ONE cell, then Save & Run All)
# Each phase cell reads these switches and no-ops if not selected. Pick a ~12h chunk:
#   A: RUN_PHASE1=True, P2A_LEVELS=['fp32','int8']            (~6h)
#   B: P2A_LEVELS=['int4']                                   (~3h)
#   C: P2B_LEVELS=['fp32','int8']                            (~7h)
#   D: P2B_LEVELS=['int4']                                   (~4h)
#   E: RUN_PHASE3=True, RUN_FIGURES=True                     (~4h)
RUN_PHASE1  = True                 # Cell 7  — no-defense baseline
P2A_LEVELS  = ['fp32', 'int8']     # Cell 8  — AT levels this batch ([] = skip)
P2B_LEVELS  = []                   # Cell 9  — AT+KD levels this batch ([] = skip)
RUN_PHASE3  = False                # Cell 10 — combined attack
RUN_FIGURES = False                # Cell 12 — paper figures

SAVE_DATASET = True
DATASET_SLUG = 'advc-results'      # -> <KAGGLE_USER>/advc-results
KAGGLE_USER  = ''                  # REQUIRED for the first-ever save; e.g. 'jmanav'

print('This batch will run:')
print('  Phase 1     :', RUN_PHASE1)
print('  Phase 2a AT :', P2A_LEVELS or 'skip')
print('  Phase 2b KD :', P2B_LEVELS or 'skip')
print('  Phase 3     :', RUN_PHASE3)
print('  Figures     :', RUN_FIGURES)
print('  Save dataset:', SAVE_DATASET, f'({KAGGLE_USER}/{DATASET_SLUG})' if KAGGLE_USER else '(user not set)')

In [ ]:
# Cell 6 — Smoke test
import torch
from models.loader import load_config, load_model
cfg = load_config('configs/base.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dataset in config:', cfg['dataset']['name'])
model = load_model('deit_small', 'fp32', cfg, device=device)
model.eval()
dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
    if hasattr(out, 'logits'):
        out = out.logits
print('Output shape:', tuple(out.shape), '(expected (2, 1000))')
del model, dummy
torch.cuda.empty_cache()
print('Smoke test passed.')

In [ ]:
# Cell 7 — Phase 1 (no-defense baseline)
import subprocess, sys, os
if not RUN_PHASE1:
    print('Phase 1 not selected in Cell 5d — skipping.')
else:
    proc = subprocess.Popen([sys.executable, 'experiments/eval_phase1.py', '--model', 'deit_small'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('\n[Phase 1] exit code', proc.returncode)

In [ ]:
# Cell 8 — Phase 2a (AT), auto-skip already-trained levels
import subprocess, sys, os, glob
DS_NAME = 'tiny-imagenet'
EPOCHS = 7
if not P2A_LEVELS:
    print('Phase 2a: no levels selected in Cell 5d (P2A_LEVELS=[]) — skipping.')
else:
    for compression in P2A_LEVELS:
        ckpt = glob.glob(f'results/checkpoints/{DS_NAME}/at/at_{compression}_epoch{EPOCHS:02d}*.pt')
        cmd = [sys.executable, 'experiments/eval_phase2_at.py', '--compression', compression]
        if ckpt:
            cmd.append('--skip-training')
            note = f'checkpoint found ({os.path.basename(ckpt[0])}) -> --skip-training (eval only)'
        else:
            note = 'no checkpoint -> training 7 epochs'
        print('=' * 60)
        print(f'Phase 2a: AT — {compression}  [{note}]')
        print('=' * 60)
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, env={**os.environ})
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait()
        print(f'[Phase 2a {compression}] exit code', proc.returncode, '\n')

In [ ]:
# Cell 9 — Phase 2b (AT+KD), auto-skip already-trained levels
import subprocess, sys, os, glob
DS_NAME = 'tiny-imagenet'
EPOCHS = 7
if not P2B_LEVELS:
    print('Phase 2b: no levels selected in Cell 5d (P2B_LEVELS=[]) — skipping.')
else:
    for compression in P2B_LEVELS:
        ckpt = glob.glob(f'results/checkpoints/{DS_NAME}/atkd/atkd_{compression}_epoch{EPOCHS:02d}*.pt')
        cmd = [sys.executable, 'experiments/eval_phase2_atkd.py', '--compression', compression]
        if ckpt:
            cmd.append('--skip-training')
            note = f'checkpoint found ({os.path.basename(ckpt[0])}) -> --skip-training (eval only)'
        else:
            note = 'no checkpoint -> training 7 epochs'
        print('=' * 60)
        print(f'Phase 2b: AT+KD — {compression}  [{note}]')
        print('=' * 60)
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, env={**os.environ})
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait()
        print(f'[Phase 2b {compression}] exit code', proc.returncode, '\n')

In [ ]:
# Cell 10 — Phase 3 (combined attack vs all defenses)
import subprocess, sys, os
if not RUN_PHASE3:
    print('Phase 3 not selected in Cell 5d — skipping.')
else:
    proc = subprocess.Popen([sys.executable, 'experiments/eval_phase3.py', '--model', 'deit_small'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('\n[Phase 3] exit code', proc.returncode)

In [ ]:
# Cell 11 — Preview results (per-dataset path)
import pandas as pd, os
DS_NAME = 'tiny-imagenet'
report = {
    'Phase 1 — No Defense': f'results/{DS_NAME}/phase1_results.csv',
    'Phase 2a — AT':        f'results/{DS_NAME}/phase2_at_results.csv',
    'Phase 2b — AT+KD':     f'results/{DS_NAME}/phase2_atkd_results.csv',
    'Phase 3 — Combined':   f'results/{DS_NAME}/phase3_results.csv',
}
cols = ['compression', 'defense', 'attack', 'clean_acc', 'robust_acc', 'asr', 'robustness_gap']
for title, path in report.items():
    print('\n' + '=' * 60 + '\n' + title + '\n' + '=' * 60)
    if not os.path.exists(path):
        print('  not yet generated:', path); continue
    df = pd.read_csv(path)
    if df.empty:
        print('  file exists but empty'); continue
    print(df[[c for c in cols if c in df.columns]].to_string(index=False))
    print(f'  {len(df)} row(s)')

In [ ]:
# Cell 12 — Paper figures
import subprocess, sys, os
DS_NAME = 'tiny-imagenet'
if not RUN_FIGURES:
    print('Figures not selected in Cell 5d — skipping.')
else:
    proc = subprocess.Popen([sys.executable, 'utils/paper_figures.py', '--n-samples', '4', '--n-eval', '200'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('\n[Figures] exit code', proc.returncode)
    figs = os.path.join('results', DS_NAME, 'figures')
    if os.path.isdir(figs):
        for f in sorted(os.listdir(figs)):
            if f != '.gitkeep':
                print(' ', f, f'{os.path.getsize(os.path.join(figs, f))/1024:.0f} KB')

In [ ]:
# Cell 13 — Persist results to a Kaggle dataset (auto create-or-version, batch-safe)
# Runs unconditionally at the end. Auto-creates advc-results on the first run and
# versions it afterwards. Failures are caught — they never fail the committed run
# (results still live in this version's Output tab).
import os, json, subprocess
RESULTS_DIR = '/kaggle/working/ADVC/results'

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr.strip()[-1500:])
    return r.returncode

if not SAVE_DATASET:
    print('SAVE_DATASET=False in Cell 5d — not persisting.')
else:
    try:
        meta_path = os.path.join(RESULTS_DIR, 'dataset-metadata.json')
        if not os.path.exists(meta_path):
            if not KAGGLE_USER:
                raise RuntimeError('First save needs KAGGLE_USER set in Cell 5d (e.g. "jmanav").')
            json.dump({'title': DATASET_SLUG, 'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
                       'licenses': [{'name': 'CC0-1.0'}]}, open(meta_path, 'w'))
            print(f'Creating dataset {KAGGLE_USER}/{DATASET_SLUG} ...')
            rc = _run(['kaggle', 'datasets', 'create', '-p', RESULTS_DIR, '-r', 'zip'])
        else:
            print('Versioning existing dataset ...')
            rc = _run(['kaggle', 'datasets', 'version', '-p', RESULTS_DIR,
                       '-m', 'session results update', '-r', 'zip'])
        print('\nSaved OK. Attach advc-results next batch so Cell 5b restores it.' if rc == 0
              else '\nWARNING: save failed (see above). Download from the Output tab. Run is fine.')
    except Exception as e:
        print('WARNING: persistence step skipped:', e)
        print('Results are still in this version Output tab — download them from there.')

---
## Running with your PC off — batch mode (Save & Run All)

The full matrix (~20 GPU-h) does not fit one 12h session, so run it in chunks. Batch mode re-runs every cell from scratch on a fresh machine and **commits nothing if it times out** — so each batch must fit ~12h and restore prior work from `advc-results`.

### Per-batch procedure
1. **Edit only Cell 5d** to select this batch's chunk. Set `KAGGLE_USER` once for the first save.
2. **Attach datasets**: Tiny-ImageNet **and** `advc-results` (from batch 2 on).
3. Confirm **GPU T4 x2**, **Internet On**, secrets `HF_TOKEN` + `KAGGLE_USERNAME`/`KAGGLE_KEY`.
4. **Save Version → Save & Run All (Commit) → Save.** Wait for *Running*, then shut down your PC.
5. When done: **Versions** tab → confirm *Complete*. Cell 13 saved a new `advc-results` version.

### Suggested chunks (one per batch)
| Batch | Cell 5d settings | Approx |
|---|---|---|
| A | `RUN_PHASE1=True`, `P2A_LEVELS=['fp32','int8']`, rest off | ~6h |
| B | `P2A_LEVELS=['int4']`, rest off | ~3h |
| C | `P2B_LEVELS=['fp32','int8']`, rest off | ~7h |
| D | `P2B_LEVELS=['int4']`, rest off | ~4h |
| E | `RUN_PHASE3=True`, `RUN_FIGURES=True`, rest off | ~4h |

### Guardrails
- **Never scope a batch beyond ~10h** — a timeout commits nothing.
- **Always attach `advc-results` from batch 2 on** — forget it and everything retrains.
- **Edits only take effect when you commit** — change Cell 5d, then Save Version.
- **No progressive-chain scheduler on free tier** — edit Cell 5d + attach dataset + Save Version per batch.

### Confirm it is NOT restarting from scratch
In the log, resumed levels show `checkpoint found -> --skip-training (eval only)` with **no** `Epoch 1/7`; recorded rows show `Resuming — N combination(s) already done`. Cell 5c prints the summary before any GPU is spent.